# 🌙 11. Lunar Image Registration & Geometric Warping Engine

**Mission Context**: Homography projective warp execution, coordinate transformation decomposition, and difference residual verification.  
**Objectives**:
- Estimate 3x3 Projective Homography from MAGSAC++ inliers.
- Decompose transformation matrix into Rotation ($^\circ$), Scale ($s_x, s_y$), and Translation ($t_x, t_y$).
- Warp source image using sub-pixel bilinear interpolation.
- Generate Registered Image, Checkerboard Overlay, and Difference Residual Map.
- Export `registered_image.png` and `transformation_matrix.npy`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.registration import RegistrationEngine
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()
gen = LunarSyntheticGenerator(size=(512, 512), seed=42)
pair = gen.generate_registered_pair(rotation_deg=12.5, scale=1.06, tx=35.0, ty=-20.0)

ref_img, src_img = pair["reference_image"], pair["source_image"]
H_gt = pair["homography_ground_truth"]

reg_engine = RegistrationEngine()
reg_res = reg_engine.register(src_img, ref_img, H_gt)
decomp = reg_res["decomposition"]

print("--- Geometric Decomposition ---")
print(f"Estimated Rotation: {decomp['rotation_deg']}°")
print(f"Estimated Scale: {decomp['scale_mean']} (Sx: {decomp['scale_x']}, Sy: {decomp['scale_y']})")
print(f"Translation Vector: ({decomp['tx']} px, {decomp['ty']} px)")


In [ ]:
# Visualize Registration Artifacts
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(reg_res["registered_image"], cmap='gray')
axes[0].set_title("Warped Registered Frame", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(reg_res["checkerboard"], cmap='gray')
axes[1].set_title("Checkerboard Alignment Verification", fontweight='bold')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(reg_res["difference_map"], cv2.COLOR_BGR2RGB))
axes[2].set_title("Absolute Difference Residuals", fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/11_registration_artifacts.png", dpi=300)
plt.show()


In [ ]:
# Export Registered Image and Transformation Matrix
os.makedirs("outputs/registered", exist_ok=True)
cv2.imwrite("outputs/registered/registered_image.png", reg_res["registered_image"])
np.save("outputs/registered/transformation_matrix.npy", H_gt)
print("Exported outputs/registered/registered_image.png and transformation_matrix.npy")
